In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tabulate import tabulate
import matplotlib.pyplot as plt
import scipy.stats as st

In [6]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
KONDINSKY_PREFIX = "K"
ISHIM_PREFIX = "I"

DATA_DIR = Path("/content/drive/MyDrive/data")
PRIVATE_DATA_DIR = DATA_DIR / "private"

# Загрузка и предобработка данных

Попробуем прочитать исходные данные

Ишимский район

In [ ]:
ishim_table = pd.read_parquet(PRIVATE_DATA_DIR / "ishim.parquet.gzip")

In [ ]:
ishim_table.dtypes

In [ ]:
ishim_table = ishim_table.to_csv('ishim.csv', index = False)

Кондинский район

In [ ]:
files_for_kondinsky = (PRIVATE_DATA_DIR / "kondinsky").glob("**/*.parquet.gzip")
dfs_for_kondinsky = []
for file in files_for_kondinsky:
  df = pd.read_parquet(file)
  dfs_for_kondinsky.append(df)
kondinsky_table = pd.concat(dfs_for_kondinsky, ignore_index=True)

In [ ]:
kondinsky_table

In [ ]:
kondinsky_table.dtypes

In [ ]:
kondinsky_table = kondinsky_table.to_csv('kondinsky.csv', index = False)

In [ ]:
kondinsky_rename_dict = {
    "№ случая": "referral_id",
    "Лок. Номер": "patient_id",
    "Дата рождения": "birthday",
    "Пол": "sex",
    "Предварительный диагноз": "diagnosis_code",
    "Наименование теста": "test_name",
    "Результат": "result",
    # Вероятней всего это не дата сдачи анализа,
    # а дата создания направления (чаще всего это может быть дата приема пациента).
    # Однако для простоты можем использовать эту колонку как дату сдачи анализа
    "Дата направления": "sampling_date",
    # "Национальность": "",
    # "№ анализа": "",
    # "Наименование анализа": "",
    # "№ теста": "",
    # "Отделение": "",
    # "Окончательный диагноз": "",
    # "№ карты": "",
    # "Тип": "",
}

ishim_rename_dict = {
    "Orders_ID": "referral_id",
    "PatientId": "patient_id",
    "Birthday": "birthday",
    "Sex": "sex",
    "DiagnosisCode": "diagnosis_code",
    "TestName": "test_name",
    "QuantResult": "result",
    "DateTimeSampling": "sampling_date",
    # "DateTime": "",
    # "RequestId": "",
    # "SampleId": "",
    # "LaboratoryId": "",
    # "OrganizationId": "",
    # "BiomaterialId": "",
    # "CountBiomaterial": "",
    # "NameUnit": "",
    # "Surname": "",
    # "Name_": "",
    # "Patronymic": "",
    # "HospitalDepartmentId": "",
    # "HospitalDepartmentName": "",
    # "DoctorId": "",
    # "DoctorFullName": "",
    # "TestId": "",
    # "Mnemonic": "",
    # "QualResult": "",
    # "UnitName": "",
    # "BoundRefer": "",
    # "Comment": "",
}


def load_kondinsky():
    files = (PRIVATE_DATA_DIR / "kondinsky").glob("**/*.parquet.gzip")
    dfs = []
    for file in files:
        df = pd.read_parquet(file)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def load_ishim():
    return pd.read_parquet(PRIVATE_DATA_DIR / "ishim.parquet.gzip")


def load_reference():
    ref_file_path = DATA_DIR / "analysis-reference-new.xlsx"

    oak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник ОАК")
    bak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник БАК")

    oak_ref["ref_analysis_name"] = "Общий анализ крови"
    bak_ref["ref_analysis_name"] = "Биохимический анализ крови"

    ref = pd.concat([oak_ref, bak_ref], ignore_index=True)

    return ref


def ishim_date_converter(column):
    return pd.to_datetime(column, format="%d.%m.%Y", errors="coerce")


def kondinsky_date_converter(column):
    return pd.to_datetime(column, dayfirst=True, errors="coerce")


# Используется для предобработки данных в конкретном районе.
# Особенности предобработки задаются через параметры
def preprocess_district_data(
    df,
    rename_dict,
    reference,
    column_in_ref,
    id_prefix,
    date_converter,
    district_name,
):
    # Удаляем ненужные столбцы и переименовываем оставшиеся
    columns_to_save = rename_dict.keys()
    columns_to_drop = set(df.columns).difference(columns_to_save)
    df.drop(columns=columns_to_drop, inplace=True)
    df.rename(columns=rename_dict, inplace=True)

    # Сразу уберем много записей, чтобы не делать ненужные join'ы (ускорит процесс)
    df.dropna(subset=["result", "referral_id"], inplace=True)

    # Заменяем название тестов на справочные
    ref_cols = [
        column_in_ref,
        "ref_analysis_name",
        "ref_test_name",
        "ref_test_mnemonic",
    ]
    df = pd.merge(df, reference[ref_cols], left_on="test_name", right_on=column_in_ref)

    # Убираем колонки left_on и right_on
    df.drop(columns=["test_name", column_in_ref], inplace=True)

    # Создаем уникальные идентификаторы, чтобы случайно не было повторов
    df["referral_id"] = id_prefix + "_" + df["referral_id"].astype(str)
    df["patient_id"] = id_prefix + "_" + df["patient_id"].astype(str)

    # Не можем конвертировать дату в объединенном датасете,
    # потому что один из форматов всегда будет считаться ошибкой. Поэтому делаем здесь,
    # с использованием внешней функции
    df["sampling_date"] = date_converter(df["sampling_date"])
    df["birthday"] = date_converter(df["birthday"])

    # Поможет различать соединенные выборки
    df["district_name"] = district_name

    return df


def preprocess_united_data_inplace(df):
    # errors='coerce' запишет NA, если функция не смогла преобразовать значение

    # Стандартизируем пол
    sex_map = {1.0: "м", 2.0: "ж", "М": "м", "Ж": "ж"}
    df["sex"] = df["sex"].map(sex_map)

    # Стандартизируем разделитель
    result = df["result"].astype("string").str.replace(",", ".")
    df["result"] = pd.to_numeric(result, errors="coerce").astype(float)

    # Посчитаем возраст код на момент сдачи
    df["age_when_sampling"] = pd.to_datetime(df["sampling_date"]).dt.year - pd.to_datetime(df["birthday"]).dt.year


    # Теперь удаляем строки, в которых что-то было где-то неправильно
    columns_to_ignore_dropna = ["ref_test_mnemonic"]
    columns_to_dropna = set(df.columns).difference(columns_to_ignore_dropna)
    df.dropna(subset=columns_to_dropna, inplace=True)

Теперь начинаем работать

In [ ]:
kondinsky_rename_dict = {
    "№ случая": "referral_id",
    "Лок. Номер": "patient_id",
    "Дата рождения": "birthday",
    "Пол": "sex",
    "Предварительный диагноз": "diagnosis_code",
    "Наименование теста": "test_name",
    "Результат": "result",
    # Вероятней всего это не дата сдачи анализа,
    # а дата создания направления (чаще всего это может быть дата приема пациента).
    # Однако для простоты можем использовать эту колонку как дату сдачи анализа
    "Дата направления": "sampling_date",
    # "Национальность": "",
    # "№ анализа": "",
    # "Наименование анализа": "",
    # "№ теста": "",
    # "Отделение": "",
    # "Окончательный диагноз": "",
    # "№ карты": "",
    # "Тип": "",
}

ishim_rename_dict = {
    "Orders_ID": "referral_id",
    "PatientId": "patient_id",
    "Birthday": "birthday",
    "Sex": "sex",
    "DiagnosisCode": "diagnosis_code",
    "TestName": "test_name",
    "QuantResult": "result",
    "DateTimeSampling": "sampling_date",
    # "DateTime": "",
    # "RequestId": "",
    # "SampleId": "",
    # "LaboratoryId": "",
    # "OrganizationId": "",
    # "BiomaterialId": "",
    # "CountBiomaterial": "",
    # "NameUnit": "",
    # "Surname": "",
    # "Name_": "",
    # "Patronymic": "",
    # "HospitalDepartmentId": "",
    # "HospitalDepartmentName": "",
    # "DoctorId": "",
    # "DoctorFullName": "",
    # "TestId": "",
    # "Mnemonic": "",
    # "QualResult": "",
    # "UnitName": "",
    # "BoundRefer": "",
    # "Comment": "",
}


def load_kondinsky():
    files = (PRIVATE_DATA_DIR / "kondinsky").glob("**/*.parquet.gzip")
    dfs = []
    for file in files:
        df = pd.read_parquet(file)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def load_ishim():
    return pd.read_parquet(PRIVATE_DATA_DIR / "ishim.parquet.gzip")


def load_reference():
    ref_file_path = DATA_DIR / "analysis-reference-new.xlsx"

    oak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник ОАК")
    bak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник БАК")

    oak_ref["ref_analysis_name"] = "Общий анализ крови"
    bak_ref["ref_analysis_name"] = "Биохимический анализ крови"

    ref = pd.concat([oak_ref, bak_ref], ignore_index=True)

    return ref


def ishim_date_converter(column):
    return pd.to_datetime(column, format="%d.%m.%Y", errors="coerce")


def kondinsky_date_converter(column):
    return pd.to_datetime(column, dayfirst=True, errors="coerce")


# Используется для предобработки данных в конкретном районе.
# Особенности предобработки задаются через параметры
def preprocess_district_data(
    df,
    rename_dict,
    reference,
    column_in_ref,
    id_prefix,
    date_converter,
    district_name,
):
    # Удаляем ненужные столбцы и переименовываем оставшиеся
    columns_to_save = rename_dict.keys()
    columns_to_drop = set(df.columns).difference(columns_to_save)
    df.drop(columns=columns_to_drop, inplace=True)
    df.rename(columns=rename_dict, inplace=True)

    # Сразу уберем много записей, чтобы не делать ненужные join'ы (ускорит процесс)
    df.dropna(subset=["result", "referral_id"], inplace=True)

    # Заменяем название тестов на справочные
    ref_cols = [
        column_in_ref,
        "ref_analysis_name",
        "ref_test_name",
        "ref_test_mnemonic",
    ]
    df = pd.merge(df, reference[ref_cols], left_on="test_name", right_on=column_in_ref)

    # Убираем колонки left_on и right_on
    df.drop(columns=["test_name", column_in_ref], inplace=True)

    # Создаем уникальные идентификаторы, чтобы случайно не было повторов
    df["referral_id"] = id_prefix + "_" + df["referral_id"].astype(str)
    df["patient_id"] = id_prefix + "_" + df["patient_id"].astype(str)

    # Не можем конвертировать дату в объединенном датасете,
    # потому что один из форматов всегда будет считаться ошибкой. Поэтому делаем здесь,
    # с использованием внешней функции
    df["sampling_date"] = date_converter(df["sampling_date"])
    df["birthday"] = date_converter(df["birthday"])

    # Поможет различать соединенные выборки
    df["district_name"] = district_name

    return df


def preprocess_united_data_inplace(df):
    # errors='coerce' запишет NA, если функция не смогла преобразовать значение

    # Стандартизируем пол
    sex_map = {1.0: "м", 2.0: "ж", "М": "м", "Ж": "ж"}
    df["sex"] = df["sex"].map(sex_map)

    # Стандартизируем разделитель
    result = df["result"].astype("string").str.replace(",", ".")
    df["result"] = pd.to_numeric(result, errors="coerce").astype(float)

    # Посчитаем возраст код на момент сдачи
    df["age_when_sampling"] = pd.to_datetime(df["sampling_date"]).dt.year - pd.to_datetime(df["birthday"]).dt.year


    # Теперь удаляем строки, в которых что-то было где-то неправильно
    columns_to_ignore_dropna = ["ref_test_mnemonic"]
    columns_to_dropna = set(df.columns).difference(columns_to_ignore_dropna)
    df.dropna(subset=columns_to_dropna, inplace=True)

In [ ]:
print("Loading analysis reference... ", end="", flush=True)
reference = load_reference()
print("done")

print("Loading data [1/2] (kondinsky)... ", end="", flush=True)
df_kondinsky = load_kondinsky()
df_kondinsky = preprocess_district_data(
    df_kondinsky,
    kondinsky_rename_dict,
    reference,
    "kondinsky_both_test_name",
    KONDINSKY_PREFIX,
    kondinsky_date_converter,
    "Кондинский район",
)
print("done")

print("Loading data [2/2] (ishim)... ", end="", flush=True)
df_ishim = load_ishim()
df_ishim = preprocess_district_data(
    df_ishim,
    ishim_rename_dict,
    reference,
    "ishim_test_name",
    ISHIM_PREFIX,
    ishim_date_converter,
    "Ишимский район",
)
print("done")

print("Unioning data... ", end="", flush=True)
df_all = pd.concat([df_kondinsky, df_ishim], ignore_index=True)
print("done")

print("Preprocessing united data... ", end="", flush=True)
preprocess_united_data_inplace(df_all)
print("done")

In [ ]:
# Описание колонок:
#
# referral_id - номер направления
# patient_id - номер пациента
#
# birthday - дата рождения пациента
# sampling_date - дата взятия анализа
# sex - пол
# age_when_sampling - возраст пациента во время взятия анализа (в целых годах; рассчитан на основе даты рождения и даты взятия анализа)
# diagnosis_code - код диагноза (МКБ-10)
#
# result - результат анализа
# ref_analysis_name - название группы анализов (либо общий анализ крови, либо биохимический анализ крови)
# ref_test_name - название конкретного анализа
# ref_test_mnemonic - сокращенное название конкретного анализа (есть не для всех)
#
# district_name - название района
df_all = df_all[:9999]

In [ ]:
# Пропущенные значения есть только для мнемоник, во всех других местах все значения есть
df_all.isna().sum()

In [ ]:
# Для всех колонок есть указаны типы данных
df_all.dtypes

## Удаляем строки с лишними и пропущенными данными

### Удаляем дублирующиеся данные

In [ ]:
# смотрим, есть ли дублирующиеся данные
df_all.duplicated()

In [ ]:
# но не факт, что дубликатов нет, поэтому их удалим
df_all = df_all.drop_duplicates()

In [ ]:
# проверяем количество строк в таблице
len(df_all)

Мы удалили дубликаты данных. Количество строк стало меньше.

### Удаляем пропущенные данные

In [ ]:
# смотрим, есть ли пропущенные данные
df_all.isna()

In [ ]:
# смотрим, есть ли пропущенные данные
df_all.isnull()

In [ ]:
# удаляем данные
df_all = df_all.dropna()

In [ ]:
df_all

# Работа с данными

<h3>diagnosis_code</h3>

In [ ]:
# выведем количество пациентов с каждым диагнозом ССЗ
diagnosis_data = df_all.loc[df_all['diagnosis_code'].str.contains('I')]['diagnosis_code'].value_counts()
diagnosis_data

In [ ]:
# слишком много, поэтому визуализируем данные по первым 5 признакам
diagnosis_data[:10].plot(kind = 'bar')
plt.xlabel('diagnosis_code')
plt.ylabel('Количество пациентов')
plt.show()

<h2>Выведем параметры общего и биохимического анализов крови</h2>

<h2>Отфильтруем таблицу по параметрам: диагноз - болезни органов кровоообращения</h2>

In [ ]:
df_cvd = df_all.loc[(df_all['diagnosis_code'].str.contains('I11'))|(df_all['diagnosis_code'].str.contains('I20'))|(df_all['diagnosis_code'].str.contains('I10'))|(df_all['diagnosis_code'].str.contains('I67'))|(df_all['diagnosis_code'].str.contains('I25'))]

In [ ]:
df_cvd

Выведем список параметров

In [ ]:
df_cvd.ref_test_mnemonic.unique()

<h2>Таблица без болезней органов кровообращения</h2>

In [ ]:
df_not_cvd = df_all.loc[df_all['diagnosis_code'].str.contains('I') == False]

In [ ]:
df_not_cvd

<h3>Параметры анализа крови</h3>

Если хб один болеет ССЗ, то заболевание есть

In [ ]:
# сортируем по patient_id
# смотрим все строки с patient_id
# если

In [ ]:
# сравним их: если пациент из списка с ССЗ есть в списке без ССЗ, то строку удаляем
for index_1, key_1 in df_not_cvd.iterrows():
    for index_2, key_2 in df_cvd.iterrows():
        if key_1['patient_id'] == key_2['patient_id']:
            print(key_1['patient_id'], key_1['diagnosis_code'])
            df_not_cvd.drop(index_1)

In [ ]:
df_cvd

In [ ]:
df_cvd.ref_test_mnemonic.unique()

In [ ]:
df_not_cvd

In [ ]:
df_not_cvd.ref_test_mnemonic.unique()

#### <h2>Описательные статистики</h2>

<h3>1.1. sex (с 5 группами ССЗ)</h3>

In [ ]:
# выведем доли мужчин и женщин
sex_cvd_data = (df_cvd['sex'].value_counts())/len(df_cvd['sex'])*100
sex_cvd_data

Визуализируем это

In [ ]:
# визуализируем данные
sex_cvd_data.plot(kind = 'bar')
plt.xlabel('sex')
plt.ylabel('Доля (в %)')
plt.title('Доли пациентов по полам')
plt.show()

<h3>1.2. sex (без ССЗ)</h3>

In [ ]:
# выведем доли мужчин и женщин
sex_not_cvd_data = (df_not_cvd['sex'].value_counts())/len(df_not_cvd['sex'])*100
sex_not_cvd_data

Визуализируем это

In [ ]:
# визуализируем данные
sex_not_cvd_data.plot(kind = 'bar')
plt.xlabel('sex')
plt.ylabel('Доля (в %)')
plt.title('Доли пациентов по полам')
plt.show()

<h3>2.1. age_when_sampling (с 5 самыми распространенными заболеваниями ССЗ)</h3>

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
age_cvd_mean = df_cvd['age_when_sampling'].mean()     # среднее
age_cvd_std = df_cvd['age_when_sampling'].std()     # стандатрное отклонение
print("Среднее значение возраста: \n", age_cvd_mean , '±', age_cvd_std, 'лет')

<h4>Проверим выборку на нормальность</h4>

<u>Гипотеза H0: возраст пациентов с ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: возраст пациентов с ССЗ имеет распределение, отличающееся от нормального.</u>

In [ ]:
from scipy.stats import shapiro
stat, p_age = shapiro(df_cvd['age_when_sampling'])
print('Вероятность по критерию Шапиро-Уилка', p_age)
if (p_age < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

<h3>2.2. age_when_sampling (без ССЗ)</h3>

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
age_not_cvd_mean = df_not_cvd['age_when_sampling'].mean()     # среднее
age_not_cvd_std = df_not_cvd['age_when_sampling'].std()     # стандатрное отклонение
print("Среднее значение возраста: \n", age_not_cvd_mean , '±', age_not_cvd_std, 'лет')

<h4>Проверим выборку на нормальность</h4>

In [ ]:
from scipy.stats import shapiro
stat, p_age = shapiro(df_not_cvd['age_when_sampling'])
print('Вероятность по критерию Шапиро-Уилка', p_age)
if (p_age < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

<u>Гипотеза H0: возраст пациентов без ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: возраст пациентов без ССЗ имеет распределение, отличающееся от нормального.</u>

Проверяем с помощью критерия Манна-Уитни

In [ ]:
import scipy.stats
m_wh = scipy.stats.mannwhitneyu(df_cvd['age_when_sampling'], df_not_cvd['age_when_sampling'])
if m_wh.pvalue < 0.05:
    print('Между средними арифметическими выборок наблюдается статистически достоверная разница')
else:
    print('Между средними арифметическими выборок статистически достоверная разница НЕ наблюдается')

Таким образом, возраст является одним из признаков, по которому можно сделать дерево решений

In [ ]:
age_not_cvd_normalized

<h3>3.1. district_name (с 5 самыми распространенными заболеваниями ССЗ)</h3>

<h3>2.4.Нормализация значений признака age_when_sampling</h3>

In [ ]:
# выведем количество пациентов в Ишимском районе и пациентов в Кондинском районе
district_data = (df_cvd['district_name'].value_counts())/len(df_cvd['district_name'])*100
district_data

In [ ]:
# визуализируем данные
district_data.plot(kind = 'bar')
plt.xlabel('district_name')
plt.ylabel('Количество пациентов')
plt.show()

<h3>3.2. district_name (без ССЗ)</h3>

In [ ]:
# выведем количество пациентов в Ишимском районе и пациентов в Кондинском районе
district_data = (df_not_cvd['district_name'].value_counts())/len(df_not_cvd['district_name'])*100
district_data

In [ ]:
# визуализируем данные
district_data.plot(kind = 'bar')
plt.xlabel('district_name')
plt.ylabel('Количество пациентов')
plt.show()

Район, возможно, также можно отнести к признакам, по которому можно классифицировать ССЗ.

<h3>Исследуем параметры анализов крови</h3>

ССЗ: 'WBC', 'MONO', 'RBC'

Не-ССЗ: 'WBC', 'MONO', 'RBC'

<h3>4.1. WBC (с 5 самыми распространенными заболеваниями ССЗ)</h3>

In [ ]:
df_cvd_wbc = df_cvd.loc[(df_cvd['ref_test_mnemonic'] == 'WBC')]

In [ ]:
df_cvd_wbc

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
wbc_cvd_mean = df_cvd_wbc['result'].mean()     # среднее
wbc_cvd_std = df_cvd_wbc['result'].std()     # стандатрное отклонение
print("Среднее значение признака WBC (количество лейкоцитов) у пациентов с 5 самыми распространенными ССЗ: \n", wbc_cvd_mean , '±', wbc_cvd_std, '* 10^9/л')

<h4>Проверим выборку на нормальность</h4>

<u>Гипотеза H0: значение признака WBC с 5 самыми распространенными ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: значение признака WBC с 5 самыми распространенными ССЗ имеет распределение, отличающееся от нормального.</u>

In [ ]:
from scipy.stats import shapiro
stat_wbc_cvd, p_wbc_cvd = shapiro(df_cvd_wbc['result'])
print('Вероятность по критерию Шапиро-Уилка', p_wbc_cvd)
if (p_wbc_cvd < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

<h3>4.2. WBC (без ССЗ) </h3>

In [ ]:
df_not_cvd_wbc = df_not_cvd.loc[(df_not_cvd['ref_test_mnemonic'] == 'WBC')]

In [ ]:
df_not_cvd_wbc

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
wbc_not_cvd_mean = df_not_cvd_wbc['result'].mean()     # среднее
wbc_not_cvd_std = df_not_cvd_wbc['result'].std()     # стандатрное отклонение
print("Среднее значение признака WBC (количество лейкоцитов) у пациентов без ССЗ: \n", wbc_not_cvd_mean , '±', wbc_not_cvd_std, '* 10^9/л')

<h4>Проверим выборку на нормальность</h4>

<u>Гипотеза H0: значение признака WBC без ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: значение признака WBC без ССЗ имеет распределение, отличающееся от нормального.</u>

In [ ]:
from scipy.stats import shapiro
stat_not_wbc_cvd, p_wbc_not_cvd = shapiro(df_not_cvd_wbc['result'])
print('Вероятность по критерию Шапиро-Уилка', p_wbc_not_cvd)
if (p_wbc_not_cvd < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

<h3>4.3. Сравнение WBC (с ССЗ) и WBC (без ССЗ)</h3>

Проверяем с помощью критерия Манна-Уитни

In [ ]:
import scipy.stats
m_wh = scipy.stats.mannwhitneyu(df_cvd_wbc['result'], df_not_cvd_wbc['result'])
if m_wh.pvalue < 0.05:
    print('Между средними арифметическими выборок наблюдается статистически достоверная разница')
else:
    print('Между средними арифметическими выборок статистически достоверная разница НЕ наблюдается')

Таким образом, WBC к признакам для дерева решений не относим

<h3>5.1. MONO (с 5 самыми распространенными заболеваниями ССЗ)</h3>

In [ ]:
df_cvd_mono = df_cvd.loc[(df_cvd['ref_test_mnemonic'] == 'MONO')]

In [ ]:
df_cvd_mono

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
mono_cvd_mean = df_cvd_mono['result'].mean()     # среднее
mono_cvd_std = df_cvd_mono['result'].std()     # стандатрное отклонение
print("Среднее значение признака MONO: \n", mono_cvd_mean , '±', mono_cvd_std, '%')

<h4>Проверим выборку на нормальность</h4>

<u>Гипотеза H0: значение признака MONO с 5 самыми распространенными ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: значение признака MONO с 5 самыми распространенными ССЗ имеет распределение, отличающееся от нормального.</u>

In [ ]:
from scipy.stats import shapiro
stat_mono_cvd, p_mono_cvd = shapiro(df_cvd_mono['result'])
print('Вероятность по критерию Шапиро-Уилка', p_mono_cvd)
if (p_mono_cvd < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

<h3>5.2. MONO (без ССЗ) </h3>

In [ ]:
df_not_cvd_mono = df_not_cvd.loc[(df_not_cvd['ref_test_mnemonic'] == 'MONO')]

In [ ]:
df_not_cvd_mono

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
mono_not_cvd_mean = df_not_cvd_mono['result'].mean()     # среднее
mono_not_cvd_std = df_not_cvd_mono['result'].std()     # стандатрное отклонение
print("Среднее значение признака MONO: \n", mono_not_cvd_mean , '±', mono_not_cvd_std, '%')

<h4>Проверим выборку на нормальность</h4>

<u>Гипотеза H0: значение признака MONO без ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: значение признака MONO без ССЗ имеет распределение, отличающееся от нормального.</u>

In [ ]:
from scipy.stats import shapiro
stat_mono_not_cvd, p_mono_not_cvd = shapiro(df_not_cvd_mono['result'])
print('Вероятность по критерию Шапиро-Уилка', p_mono_not_cvd)
if (p_mono_not_cvd < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

Проверяем по критерию Манна-Уитни

In [ ]:
import scipy.stats
m_wh = scipy.stats.mannwhitneyu(df_cvd_mono['result'], df_not_cvd_mono['result'])
if m_wh.pvalue < 0.05:
    print('Между средними арифметическими выборок наблюдается статистически достоверная разница')
else:
    print('Между средними арифметическими выборок статистически достоверная разница НЕ наблюдается')

Таким образом, MONO к признакам для дерева решений не относим

<h3>6.1. RBC (с 5 самыми распространенными заболеваниями ССЗ)</h3>

Таблица значений признака RBC (количества эритроцитов) по пациентам с 5 самыми распространенными заболеваниями ССЗ

In [ ]:
df_cvd_rbc = df_cvd.loc[(df_cvd['ref_test_mnemonic'] == 'RBC')]

In [ ]:
df_cvd_rbc

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
rbc_cvd_mean = df_cvd_rbc['result'].mean()     # среднее
rbc_cvd_std = df_cvd_rbc['result'].std()     # стандатрное отклонение
print("Среднее значение признака RBC (количество эритроцитов) у пациентов с 5 самыми распространенными ССЗ: \n", rbc_cvd_mean , '±', rbc_cvd_std, '* 10^12/л')

<h4>Проверим выборку на нормальность</h4>

<u>Гипотеза H0: значение признака RBC с 5 самыми распространенными ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: значение признака RBC с 5 самыми распространенными ССЗ имеет распределение, отличающееся от нормального.</u>

In [ ]:
from scipy.stats import shapiro
stat_rbc_cvd, p_rbc_cvd = shapiro(df_cvd_rbc['result'])
print('Вероятность по критерию Шапиро-Уилка', p_rbc_cvd)
if (p_rbc_cvd < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

<h3>6.2. RBC (без ССЗ)</h3>

In [ ]:
df_not_cvd_rbc = df_not_cvd.loc[(df_not_cvd['ref_test_mnemonic'] == 'RBC')]

In [ ]:
df_not_cvd_rbc

<h4>Среднее арифметическое и стандартное отклонение</h4>

In [ ]:
rbc_not_cvd_mean = df_not_cvd_rbc['result'].mean()     # среднее
rbc_not_cvd_std = df_not_cvd_rbc['result'].std()     # стандатрное отклонение
print("Среднее значение признака RBC (количество эритроцитов) у пациентов без ССЗ: \n", rbc_not_cvd_mean , '±', rbc_not_cvd_std, '* 10^12/л')

Надо сжать выборку (убрать до 3 сигма и после 3 сигма)

In [ ]:
df_not_cvd_rbc = df_not_cvd_rbc[df_not_cvd_rbc['result'] <= df_not_cvd_rbc['result'].mean()+2.8*df_not_cvd_rbc['result'].std()]
df_not_cvd_rbc = df_not_cvd_rbc[df_not_cvd_rbc['result'] >= df_not_cvd_rbc['result'].mean()-2.8*df_not_cvd_rbc['result'].std()]


Делаем среднее и сигму снова

In [ ]:
rbc_not_cvd_mean = df_not_cvd_rbc['result'].mean()     # среднее
rbc_not_cvd_std = df_not_cvd_rbc['result'].std()     # стандатрное отклонение
print("Среднее значение признака RBC (количество эритроцитов) у пациентов без ССЗ: \n", rbc_not_cvd_mean , '±', rbc_not_cvd_std, '* 10^12/л')

<h4>Проверим выборку на нормальность</h4>

<u>Гипотеза H0: значение признака RBC без ССЗ имеет распределение, не отличающееся от нормального.</u><br>
<u>Гипотеза Ha: значение признака RBC без ССЗ имеет распределение, отличающееся от нормального.</u>

In [ ]:
from scipy.stats import shapiro
stat_rbc_not_cvd, p_rbc_not_cvd = shapiro(df_not_cvd_rbc['result'])
print('Вероятность по критерию Шапиро-Уилка', p_rbc_not_cvd)
if (p_rbc_not_cvd < 0.05):
    print("Гипотеза о нормальном распределении возраста пациентов с ССЗ отклонена")
else:
    print("Нет оснований отклонить гипотезу о нормальном распределении возраста пациентов с ССЗ")

Проверяем по критерию Манна-Уитни

In [ ]:
import scipy.stats
m_wh = scipy.stats.mannwhitneyu(df_cvd_rbc['result'], df_not_cvd_rbc['result'])
if m_wh.pvalue < 0.05:
    print('Между средними арифметическими выборок наблюдается статистически достоверная разница')
else:
    print('Между средними арифметическими выборок статистически достоверная разница НЕ наблюдается')

Таким образом, признак 'RBC' (количество эритроцитов) является одним из признаков, по которому можно сделать дерево решений. Можно нормализовать

<h3>6.4.Нормализация значений признака RBC</h3>

<h3>6.4.1. Нормализация значений признака RBC у пациентов с 5 самыми распространенными ССЗ в выборке</h3>

In [ ]:
# сделаем список
rbc_cvd_normalized = []
# добавим значения в список
for el in df_cvd_rbc['result']:
    rbc_cvd_normalized.append((el-df_cvd_rbc['result'].mean())/df_cvd_rbc['result'].std())

In [ ]:
rbc_cvd_normalized

<h3>6.4.2. Нормализация значений признака RBC без ССЗ</h3>

In [ ]:
# сделаем список
rbc_not_cvd_normalized = []
# добавим значения в список
for el in df_not_cvd_rbc['result']:
    rbc_not_cvd_normalized.append((el-df_not_cvd_rbc['result'].mean())/df_not_cvd_rbc['result'].std())

In [ ]:
rbc_not_cvd_normalized

<h3>6.5. Нормализация значений age_when_sampling (с ССЗ)</h3>

In [ ]:
# сделаем список
age_cvd_normalized = []
# добавим значения в список
for el in df_cvd_rbc['age_when_sampling']:
    age_cvd_normalized.append((el-df_cvd_rbc['age_when_sampling'].mean())/df_cvd_rbc['age_when_sampling'].std())

In [ ]:
age_cvd_normalized

<h3>6.5.2. Нормализация значений age_when_sampling (без ССЗ)</h3>

In [ ]:
# сделаем список

age_not_cvd_normalized = []

# добавим значения в список

for el in df_not_cvd_rbc['age_when_sampling']:

    age_not_cvd_normalized.append((el-df_not_cvd_rbc['age_when_sampling'].mean())/df_not_cvd_rbc['age_when_sampling'].std())

In [ ]:
age_not_cvd_normalized

<h2>7. ДЕРЕВО РЕШЕНИЙ</h2>

Пусть в исходных признаках (X) будут ‘age_when_sampling’, ’‘result’,
а целевым признаком (Y) будет 'diagnosis_code'

In [ ]:
# сделаем массивы для обучения и тестирования
# перемешаем выборку X
import random
X = np.array([age_cvd_normalized + age_not_cvd_normalized, rbc_cvd_normalized + rbc_not_cvd_normalized])
y = np.array(pd.concat([df_cvd_rbc['diagnosis_code'], df_not_cvd_rbc['diagnosis_code']]))

In [ ]:
X_new = []
for i in range(min(len(age_cvd_normalized + age_not_cvd_normalized), len(rbc_cvd_normalized + rbc_not_cvd_normalized))):
    X_new.append([X[0][i], X[1][i]])

In [ ]:
X_new

In [ ]:
y

In [ ]:
# перемешаем
random.shuffle(X_new) # не получается перемешать

In [ ]:
print(X_new)

In [ ]:
# 1. Разделим выборку на 70 - 30
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X_new, y, test_size = 0.3)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# делаем дерево решений
clf = DecisionTreeClassifier(max_leaf_nodes = 2, random_state = int)
# обучим
clf.fit(X_train, y_train)
# проиллюстрируем
tree.plot_tree(clf)
plt.show()